In [ ]:
from mlwpy import *

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(-5, 5)
ys = xs**2

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(xs, ys)

# Highlight a specific point on the curve
pt_x, pt_y = 3, 3**2
ax.plot(pt_x, pt_y, 'ro')

# Plot a tangent line at that point
line_xs = pt_x + np.array([-2, 2])
line_ys = pt_y + (line_xs - pt_x) * (2 * pt_x)  # derivative of x^2 is 2x

ax.plot(line_xs, line_ys, 'r-')
ax.set_xlabel('weight')
ax.set_ylabel('cost')
plt.tight_layout()
plt.show()


In [ ]:
weights = np.linspace(-5, 5)
costs = weights**2

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(weights, costs, 'b')

weight_min = 3
step_size = 0.25

for i in range(10):
    cost_at_min = weight_min**2
    ax.plot(weight_min, cost_at_min, 'ro')
    slope_at_min = 2 * weight_min
    weight_min = weight_min - step_size * slope_at_min

ax.set_xlabel('weight value')
ax.set_ylabel('cost')

print("Approximate location of blue graph minimum:", weight_min)


In [ ]:
from scipy.optimize import fmin as magical_minimum_finder

def f(x):
    return x**2

magical_minimum_finder(f, [3], disp=False)


In [ ]:
linreg_ftrs_p1 = np.c_[np.arange(10), np.ones(10)]  # +1 trick in data
true_wgts = m, b = w_1, w_0 = 3, 2
linreg_tgt = rdot(true_wgts, linreg_ftrs_p1)

linreg_table = pd.DataFrame(linreg_ftrs_p1, columns=['ftr_1', 'ones'])
linreg_table['tgt'] = linreg_tgt
linreg_table[:3]


In [ ]:
def linreg_model(weights, ftrs):
    return rdot(weights, ftrs)

def linreg_loss(predicted, actual):
    errors = predicted - actual
    return np.dot(errors, errors)  # sum-of-squares

def no_penalty(_weights):
    return 0.0


In [ ]:
def make_cost(ftrs, tgt,
              model_func, loss_func,
              c_tradeoff, complexity_penalty):
    """build an optimization problem from data, model, loss, penalty"""
    def cost(weights):
        return (loss_func(model_func(weights, ftrs), tgt) +
                c_tradeoff * complexity_penalty(weights))
    return cost


In [ ]:
# build linear regression optimization problem
linreg_cost = make_cost(linreg_ftrs_p1, linreg_tgt,
                        linreg_model, linreg_loss,
                        0.0, no_penalty)

learned_wgts = magical_minimum_finder(linreg_cost, [5, 5], disp=False)

print(" true weights:", true_wgts)
print("learned weights:", learned_wgts)


In [ ]:
def l1_penalty(weights):
    return np.abs(weights).sum()

def l2_penalty(weights):
    return np.dot(weights, weights)


In [ ]:
# linear regression with L1 regularization (lasso regression)
linreg_L1_pen_cost = make_cost(linreg_ftrs_p1, linreg_tgt,
                               linreg_model, linreg_loss,
                               1.0, l1_penalty)

learned_wgts = magical_minimum_finder(linreg_L1_pen_cost, [5, 5], disp=False)

print(" true weights:", true_wgts)
print("learned weights:", learned_wgts)


In [ ]:
# linear regression with L2 regularization (ridge regression)
linreg_l2_pen_cost = make_cost(linreg_ftrs_p1, linreg_tgt,
                               linreg_model, linreg_loss,
                               1.0, l2_penalty)

learned_wgts = magical_minimum_finder(linreg_l2_pen_cost, [5, 5], disp=False)

print(" true weights:", true_wgts)
print("learned weights:", learned_wgts)


In [ ]:
logreg_ftr = np.random.uniform(5, 15, size=(100,))
true_wgts = m, b = -2, 20
line_of_logodds = m * logreg_ftr + b
prob_at_x = np.exp(line_of_logodds) / (1 + np.exp(line_of_logodds))

logreg_tgt = np.random.binomial(1, prob_at_x, len(logreg_ftr))
logreg_ftrs_p1 = np.c_[logreg_ftr, np.ones_like(logreg_ftr)]

logreg_table = pd.DataFrame(logreg_ftrs_p1, columns=['ftr_1', 'ones'])
logreg_table['tgt'] = logreg_tgt

display(logreg_table.head())


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(logreg_ftr, prob_at_x, 'r.')
ax.scatter(logreg_ftr, logreg_tgt, c=logreg_tgt)


In [ ]:
# for logistic regression
def logreg_model(weights, ftrs):
    return rdot(weights, ftrs)

def logreg_loss_01(predicted, actual):
    return np.sum(-predicted * actual + np.log(1 + np.exp(predicted)))


In [ ]:
logreg_cost = make_cost(logreg_ftrs_p1, logreg_tgt,
                        logreg_model, logreg_loss_01,
                        0.0, no_penalty)

learned_wgts = magical_minimum_finder(logreg_cost, [5, 5], disp=False)

print(" true weights:", true_wgts)
print("learned weights:", learned_wgts)


In [ ]:
# logistic regression with penalty
logreg_pen_cost = make_cost(logreg_ftrs_p1, logreg_tgt,
                            logreg_model, logreg_loss_01,
                            0.5, l1_penalty)

learned_wgts = magical_minimum_finder(logreg_pen_cost, [5, 5], disp=False)

print(" true weights:", true_wgts)
print("learned weights:", learned_wgts)


In [ ]:
def binary_to_pm1(b):
    """map {0,1} or {False,True} to {-1, +1}"""
    return (b * 2) - 1

binary_to_pm1(0), binary_to_pm1(1)


In [ ]:
# for logistic regression
def logreg_model(weights, ftrs):
    return rdot(weights, ftrs)

def logreg_loss_pm1(predicted, actual):
    return np.sum(np.log(1 + np.exp(-predicted * actual)))


In [ ]:
logreg_cost = make_cost(logreg_ftrs_p1, binary_to_pm1(logreg_tgt),
                        logreg_model, logreg_loss_pm1,
                        0.0, no_penalty)

learned_wgts = magical_minimum_finder(logreg_cost, [5, 5], disp=False)

print(" true weights:", true_wgts)
print("learned weights:", learned_wgts)


In [ ]:
def predict_with_logreg_weights_to_pm1(w_hat, x):
    prob = 1 / (1 + np.exp(rdot(w_hat, x)))
    thresh = prob < 0.5
    return binary_to_pm1(thresh)

preds = predict_with_logreg_weights_to_pm1(learned_wgts, logreg_ftrs_p1)
print(metrics.accuracy_score(preds, binary_to_pm1(logreg_tgt)))


In [ ]:
# for SVC
def hinge_loss(predicted, actual):
    hinge = np.maximum(1 - predicted * actual, 0.0)
    return np.sum(hinge)

def predict_with_svm_weights(w_hat, x):
    return np.sign(rdot(w_hat, x)).astype(int)


In [ ]:
svm_ftrs = logreg_ftrs_p1
svm_tgt = binary_to_pm1(logreg_tgt)  # svm "demands" +/- 1

svc_cost = make_cost(svm_ftrs, svm_tgt, rdot,
                     hinge_loss, 0.0, no_penalty)

learned_weights = magical_minimum_finder(svc_cost, [5, 5], disp=False)
preds = predict_with_svm_weights(learned_weights, svm_ftrs)

print('no penalty accuracy:', metrics.accuracy_score(preds, svm_tgt))


In [ ]:
# svc with penalty
svc_pen_cost = make_cost(svm_ftrs, svm_tgt, rdot,
                         hinge_loss, 1.0, l1_penalty)

learned_weights = magical_minimum_finder(svc_pen_cost, [5, 5], disp=False)
preds = predict_with_svm_weights(learned_weights, svm_ftrs)

print('accuracy with penalty:', metrics.accuracy_score(preds, svm_tgt))


In [ ]:
#Here's where we go off script and replace TF with Torch, I only wish I'd thought of this sooner
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
class TorchLinearRegression(nn.Module):
    def __init__(self, n_ftrs):
        super().__init__()
        self.linear = nn.Linear(n_ftrs, 1)  # includes bias by default

    def forward(self, x):
        return self.linear(x)


In [ ]:
# remove bias column; PyTorch Linear includes bias by default
linreg_ftrs = linreg_ftrs_p1[:, 0]

# convert to tensors
X = torch.tensor(linreg_ftrs, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(linreg_tgt, dtype=torch.float32).unsqueeze(1)

# instantiate model
linreg_nn = TorchLinearRegression(1)
criterion = nn.MSELoss()
optimizer = optim.SGD(linreg_nn.parameters(), lr=0.01)

# training loop
for _ in range(1000):
    optimizer.zero_grad()
    output = linreg_nn(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

# predictions and evaluation
with torch.no_grad():
    preds = linreg_nn(X)
    mse = metrics.mean_squared_error(preds.numpy(), linreg_tgt)
    print("Training MSE: {:5.4f}".format(mse))


In [ ]:
loss_history = []

for _ in range(1000):
    optimizer.zero_grad()
    output = linreg_nn(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

print(loss_history[:5])


In [ ]:
class TorchLogisticRegression(nn.Module):
    def __init__(self, n_ftrs):
        super().__init__()
        self.linear = nn.Linear(n_ftrs, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# prepare tensors
X = torch.tensor(logreg_ftr, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(logreg_tgt, dtype=torch.float32).unsqueeze(1)

# model, loss, optimizer
logreg_nn = TorchLogisticRegression(1)
criterion = nn.BCELoss()
optimizer = optim.SGD(logreg_nn.parameters(), lr=0.01)

# train
for _ in range(1000):
    optimizer.zero_grad()
    output = logreg_nn(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

# evaluate
with torch.no_grad():
    probs = logreg_nn(X)
    preds = (probs > 0.5).int().squeeze()
    print('accuracy:', metrics.accuracy_score(preds.numpy(), logreg_tgt))


In [ ]:
# Since PyMC3 is not supported on Python 3.13, we can use sklearn to approximate the ideas in the text. While sklearn provides fast point estimates optimized for prediction, it does not offer uncertainty quantification. statsmodels produces similar estimates but adds statistical inference tools like confidence intervals and p-values, making the implicit variability in predictions more explicit. PyMC, by contrast, models full posterior distributions and captures uncertainty more completely, but it isn’t currently compatible with Python 3.13.

In [ ]:
from sklearn.linear_model import LinearRegression

X = linreg_table[['ftr_1']].values  # exclude bias column
y = linreg_table['tgt'].values

model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Slope:", model.coef_[0])


In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# mimic y = m * x + b using sklearn
X = linreg_table[['ftr_1']].values
y = linreg_table['tgt'].values

model = LinearRegression()
model.fit(X, y)

intercept = model.intercept_
ftr_1_wgt = model.coef_[0]

# residuals and estimated standard deviation
residuals = y - model.predict(X)
sd = np.std(residuals, ddof=1)  # sample std dev

print("Intercept:", round(intercept, 4))
print("ftr_1_wgt:", round(ftr_1_wgt, 4))
print("sd:", round(sd, 4))


In [ ]:
# design matrix and target
X = linreg_table[['ftr_1']].values
y = linreg_table['tgt'].values

# fit model
model = LinearRegression()
model.fit(X, y)

# extract coefficients
intercept = model.intercept_
ftr_1_wgt = model.coef_[0]

# estimate residual standard deviation (similar to noise term in PyMC)
residuals = y - model.predict(X)
sd = np.std(residuals, ddof=1)

# summarize
print("mean")
print("Intercept  ", f"{intercept:.4f}")
print("ftr_1_wgt  ", f"{ftr_1_wgt:.4f}")
print("sd         ", f"{sd:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

# Simulated or real input features and targets
X = logreg_ftr.reshape(-1, 1)  # assumes logreg_ftr is already a NumPy array
y = logreg_tgt                 # assumes this is 1D NumPy array

# Bootstrap
n_samples = 1000
intercepts = []
coefs = []
residual_sds = []

for _ in range(n_samples):
    indices = np.random.choice(len(X), size=len(X), replace=True)
    X_sample = X[indices]
    y_sample = y[indices]

    model = LinearRegression().fit(X_sample, y_sample)
    preds = model.predict(X_sample)
    residuals = y_sample - preds

    intercepts.append(model.intercept_)
    coefs.append(model.coef_[0])
    residual_sds.append(np.std(residuals))

# Now the plot from your message will work:
fig, axes = plt.subplots(3, 1, figsize=(8, 9))

for ax, data, title in zip(
    axes,
    [intercepts, coefs, residual_sds],
    ["Intercept", "ftr_1", "sd"]
):
    data = np.asarray(data)
    data_min = np.min(data)
    data_max = np.max(data)

    if data_max - data_min > 0:
        bins = min(30, 10)
        ax.hist(data, bins=bins, color='lightgray', edgecolor='black')
        sns.kdeplot(data, ax=ax, color='steelblue')
    else:
        ax.text(0.5, 0.5, 'Too little variance for histogram',
                ha='center', va='center', transform=ax.transAxes)

    ax.set_title(title)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.linear_model import LogisticRegression

X = logreg_table[['ftr_1']].values
y = logreg_table['tgt'].values

model = LogisticRegression()
model.fit(X, y)

intercept = model.intercept_[0]
coef = model.coef_[0][0]

print("Intercept:", intercept)
print("ftr_1 coefficient:", coef)


In [ ]:
from sklearn.utils import resample


# Bootstrapping
n_iter = 1000
intercepts = []
coefs = []

X = logreg_table[['ftr_1']].values
y = logreg_table['tgt'].values

for _ in range(n_iter):
    X_resampled, y_resampled = resample(X, y)
    model = LogisticRegression()
    model.fit(X_resampled, y_resampled)
    intercepts.append(model.intercept_[0])
    coefs.append(model.coef_[0][0])

# DataFrame to mimic PyMC trace
df_trace = pd.DataFrame({
    'Intercept': intercepts,
    'ftr_1': coefs
})

# Joint KDE plot
sns.jointplot(x='ftr_1', y='Intercept', data=df_trace,
              kind='kde', fill=True, height=4)
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score

# Prepare data
X = torch.tensor(logreg_ftr, dtype=torch.float32).reshape(-1, 1)
y = torch.tensor(logreg_tgt, dtype=torch.long)

# Define model (No Softmax!)
model = nn.Sequential(
    nn.Linear(1, 2)  # 1 feature -> 2 output logits
)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
for epoch in range(1000):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

# Predict and evaluate
preds = model(X).argmax(dim=1).numpy()
print(accuracy_score(y.numpy(), preds))
